# MCP Client 연습: Tool 목록 조회와 호출

- 서버 코드는 `server.py`에 있어요. **노트북에서 서버를 실행하지 않아요.**
- 이 노트북(클라이언트)이 `server.py`를 별도 프로세스로 실행하고, 거기에 STDIO로 접속해요.
- mcp 2.x 기준이에요. (`FastMCP` → `MCPServer`, `inputSchema` → `input_schema`)

In [1]:
import os
import sys
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 노트북과 같은 폴더의 server.py (Jupyter는 보통 노트북 폴더가 현재 경로)
SERVER_PATH = Path.cwd() / "server.py"
print("server.py 존재:", SERVER_PATH.exists())

# 어떤 서버를 어떻게 실행할지: 현재 커널의 파이썬으로 server.py 실행
server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

server.py 존재: True


In [2]:
# MCP Protocol: Tool 목록 요청
async def list_tools():
    # Jupyter의 sys.stderr는 실제 파일이 아니라서 서버 로그는 버림
    with open(os.devnull, "w") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await session.list_tools()


# 노트북에서는 asyncio.run() 대신 바로 await
tools = await list_tools()

print("사용 가능한 MCP Tool")
for tool in tools.tools:
    print(f"- {tool.name}: {tool.description}")

사용 가능한 MCP Tool
- get_weather: 도시의 날씨를 조회한다.
- multiply: 두 정수를 곱한다.


In [3]:
# 각 Tool의 입력 스키마: @mcp.tool()이 타입 힌트로 만든 것 -> 나중에 LLM tool calling에 넘어가는 정보
for tool in tools.tools:
    print(tool.name, tool.input_schema)

get_weather {'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'type': 'object', 'title': 'get_weatherArguments'}
multiply {'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object', 'title': 'multiplyArguments'}


In [4]:
# MCP Protocol: Tool 호출
async def call_tool(name, arguments):
    with open(os.devnull, "w") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(name, arguments)
                return result.content[0].text


print(await call_tool("get_weather", {"city": "서울"}))
print(await call_tool("multiply", {"a": 23, "b": 45}))

서울의 날씨는 맑음, 20도
1035
